# Задание 1
Реализуйте функцию `sliding_window_attention`, которая принимает тензоры `Q, K, V` и размер окна `window_size`, возвращает результат внимания с маской, как в примере выше.

Проверьте работу на случайных данных: `seq_len=10, batch=2, d_model=16` и `window_size=2`. Убедитесь, что выход имеет форму `(seq_len, batch, d_model)`.

In [5]:
import torch, torch.nn.functional as F

def sliding_window_attention(Q, K, V, window_size):
    # Q,K,V формы (seq_len, batch, d_model)
    seq_len = Q.size(0)
    batch = Q.size(1)

    # Создаём маску формата (seq_len, seq_len) с -inf для запрещённых связей
    mask = torch.full((seq_len, seq_len), float('-inf'))
    for i in range(seq_len):
        left = max(0, i - window_size)
        right = min(seq_len, i + window_size + 1)
        mask[i, left:right] = 0

    # Переставляем размерности, чтобы батч был первым
    Q = Q.transpose(0, 1)
    K = K.transpose(0, 1)
    V = V.transpose(0, 1)
    
    # Изменяем размер маски для соответствия батча размеру Q (shape: [batch, seq_len, seq_len])
    mask = mask.unsqueeze(0).expand(batch, -1, -1)

    # Используем scaled_dot_product_attention с маской
    attn_out = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)

    # Возвращаем результат обратно в исходном порядке: (seq_len, batch, d_model)
    attn_out = attn_out.transpose(0, 1)

    return attn_out, mask

# Проверка маскированного внимания:
seq_len, batch, d_model = 10, 2, 16
Q = torch.rand(seq_len, batch, d_model)
out, mask = sliding_window_attention(Q, Q, Q, window_size=2)
print(out.shape)

# Дополнительная проверка визуализацией
print("\nПример маски для позиции 5 (первый батч):")
print(mask[0, 5].detach().numpy().round(1))


torch.Size([10, 2, 16])

Пример маски для позиции 5 (первый батч):
[-inf -inf -inf   0.   0.   0.   0.   0. -inf -inf]


# Задание 2
Датасет содержит:
- train — 25 000 рецензий, помеченных как положительные или отрицательные;
- test — 25 000 рецензий для проверки.

Загрузите и прочтите датасет IMDb, 1000 примеров для тренировки и 500 для теста, выведите первые 3 рецензии и их метки. Убедитесь, что данные корректно загружены и отображаются.

In [35]:
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer,BertConfig, BertForSequenceClassification
from torch.optim import AdamW
from tqdm.auto import tqdm

dataset = load_dataset("imdb")
# Выберем 1000 примеров для обучения и 500 для теста

train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

# Вывод первых 3 рецензий и меток
for i in range(3):
    print(f"Рецензия {i+1}: {train_dataset[i]['text']}")
    print(f"Метка {i+1}: {train_dataset[i]['label']}\n")


Рецензия 1: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...
Метка 1: 1

Рецензия 2: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub your t

# Задание 3
Создайте токенизатор BERT, реализуйте функцию tokenize_batch, которая принимает батч рецензий и возвращает токенизированные данные. 

Реализуйте функцию `create_dataloader(dataset, batch_size=8, shuffle=True)`, которая:
- Использует `torch.utils.data.DataLoader`.
- В качестве `collate_fn` принимает вашу `tokenize_batch`.
- Возвращает объект `DataLoader` для переданного dataset.

In [31]:
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased', ignore_mismatches=True)

def tokenize_batch(batch):
    texts  = [x['text'] for x in batch]
    labels = [x['label'] for x in batch]
    encoding = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    )
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': torch.tensor(labels)
    }

def create_dataloader(dataset, batch_size=8, shuffle=True):
    return DataLoader(dataset, collate_fn=tokenize_batch, batch_size=batch_size, shuffle=shuffle)

train_loader = create_dataloader(train_dataset)
test_loader  = create_dataloader(test_dataset, shuffle=False) 

# Задание 4

Настройте устройство (GPU), загрузите конфигурацию BERT с двумя метками и создайте модель `BertForSequenceClassification`. Затем настройте оптимизатор `AdamW` с `learning rate 2e-5`.

In [38]:
# Устройство: GPU, если доступна
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CUDA is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

# Конфигурация
config = BertConfig.from_pretrained('google-bert/bert-base-uncased', num_labels=2)

model = BertForSequenceClassification.from_pretrained(
    'google-bert/bert-base-uncased', config=config
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

CUDA is available: True
CUDA device: Tesla T4


/home/ubuntu/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3763.27it/s]


In [40]:
epochs = 20
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        labels = batch['labels'].to(device)
        inputs = {
            "input_ids": batch['input_ids'].to(device),
            "attention_mask": batch['attention_mask'].to(device),
            "labels": labels,
        }

        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        preds         =  outputs.logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
    
    train_loss = total_loss / len(train_loader)
    train_acc  = total_correct / total_samples
    
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch} [Eval]"):
            labels = batch['labels'].to(device)
            inputs = {
                "input_ids": batch['input_ids'].to(device),
                "attention_mask": batch['attention_mask'].to(device),
                "labels": labels,
            }
            outputs = model(**inputs)
            loss = outputs.loss
            total_loss    += loss.item()
            preds         =  outputs.logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
    
    val_loss = total_loss / len(test_loader)
    val_acc  = total_correct / total_samples

    print(f"\nEpoch {epoch} results:")
    print(f"  Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
    print(f"  Eval : loss={val_loss:.4f}, acc={val_acc:.4f}\n")

print(f"Final Eval Accuracy: {val_acc:.4f}") 

Epoch 1 [Eval]: 100%|██████████| 63/63 [00:06<00:00,  9.49it/s]



Epoch 1 results:
  Train: loss=0.0863, acc=0.9740
  Eval : loss=0.4212, acc=0.8840



Epoch 2 [Eval]: 100%|██████████| 63/63 [00:06<00:00,  9.44it/s]



Epoch 2 results:
  Train: loss=0.0473, acc=0.9890
  Eval : loss=0.3357, acc=0.8940



Epoch 3 [Eval]: 100%|██████████| 63/63 [00:06<00:00,  9.39it/s]



Epoch 3 results:
  Train: loss=0.0217, acc=0.9950
  Eval : loss=0.5226, acc=0.8680



Epoch 4 [Eval]: 100%|██████████| 63/63 [00:06<00:00,  9.37it/s]



Epoch 4 results:
  Train: loss=0.0466, acc=0.9880
  Eval : loss=0.4208, acc=0.9000



Epoch 5 [Train]:  48%|████▊     | 60/125 [00:20<00:22,  2.87it/s]


KeyboardInterrupt: 